# LangChain & LangGraph

## Install Dependencies

In [1]:
!pip install -q -U langchain langgraph langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.2 MB/s eta 0:00:00


## LangChain

### Set LLM Key

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

### LangChain: Imports

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

### LangChain: Create Tool

In [4]:
@tool
def get_weather(city: str) -> str:
    """
    Simple dummy weather lookup tool.
    The agent can call this tool when the user asks about weather.
    """
    fake_weather_data = {
        "new york": "Sunny, 72F",
        "seattle": "Cloudy, 61F",
        "miami": "Hot, 88F",
        "chicago": "Windy, 65F",
    }

    return fake_weather_data.get(
        city.lower(),
        f"No weather data found for {city}."
    )

### LangChain: Create Agent


In [5]:
llm = ChatOpenAI(model="gpt-4o-mini")

langchain_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=(
        "You are a helpful assistant. "
        "Use tools when needed. "
        "Keep answers clear and concise."
    ),
)


### LangChain: Invoke Agent

In [6]:
langchain_response = langchain_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the weather in Seattle?"
        }
    ]
})

print("=================================================")
print("LANGCHAIN AGENT OUTPUT")
print("=================================================")
print(langchain_response)
print("\n")

LANGCHAIN AGENT OUTPUT
{'messages': [HumanMessage(content='What is the weather in Seattle?', additional_kwargs={}, response_metadata={}, id='0c278c0e-b592-45a5-8bf4-844f16357497'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 80, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_888e567758', 'id': 'chatcmpl-DUH9BishFKNA1pJRilrxHP8FPBwWy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d884b-2fb7-7e52-aec5-ed7d69df1d75-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Seattle'}, 'id': 'call_vs0jzYCD4mNhjOXZ43JClb52', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i

## LangGraph

Components:
- shared state
- nodes
- edges
- conditional routing

### LangGraph: Imports

In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

### LangGraph: Shared Graph State

In [8]:
# Define the shared state used across the graph
# Each node can read and update this state

class GraphState(TypedDict):
    user_input: str
    category: str
    result: str

### LangGraph: Nodes

In [9]:
# Node 1: classify the request
# This node inspects the user input and labels it

def classify_node(state: GraphState) -> GraphState:
    """
    Determine whether the user is asking about weather
    or something more general.
    """
    text = state["user_input"].lower()

    if "weather" in text:
        category = "weather"
    else:
        category = "general"

    return {
        **state,
        "category": category
    }

In [10]:
# Node 2a: handle weather questions

def weather_node(state: GraphState) -> GraphState:
    """
    Return a weather-specific response.
    """
    text = state["user_input"].lower()

    if "seattle" in text:
        answer = "The weather in Seattle is cloudy, 61F."
    elif "miami" in text:
        answer = "The weather in Miami is hot, 88F."
    elif "new york" in text:
        answer = "The weather in New York is sunny, 72F."
    else:
        answer = "I could not find the weather location."

    return {
        **state,
        "result": answer
    }

In [11]:
# Node 2b: handle general questions

def general_node(state: GraphState) -> GraphState:
    """
    Return a generic response for non-weather questions.
    """
    return {
        **state,
        "result": f"General response: {state['user_input']}"
    }

In [12]:
# Conditional routing function
# This decides which node to run after classification

def route_after_classification(state: GraphState) -> str:
    """
    Route to the correct next node based on category.
    """
    if state["category"] == "weather":
        return "weather_node"
    return "general_node"

### LangGraph: Build Graph Workflow

In [13]:
graph_builder = StateGraph(GraphState)

# Add nodes
graph_builder.add_node("classify_node", classify_node)
graph_builder.add_node("weather_node", weather_node)
graph_builder.add_node("general_node", general_node)

# Add starting edge
graph_builder.add_edge(START, "classify_node")

# Add conditional edges
graph_builder.add_conditional_edges(
    "classify_node",
    route_after_classification,
    {
        "weather_node": "weather_node",
        "general_node": "general_node"
    }
)

# Add ending edges
graph_builder.add_edge("weather_node", END)
graph_builder.add_edge("general_node", END)

# Compile the graph
langgraph_app = graph_builder.compile()

### LangGraph: Invoke Graph Workflow

In [14]:
langgraph_output_1 = langgraph_app.invoke({
    "user_input": "What is the weather in Seattle?",
    "category": "",
    "result": ""
})

print("=================================================")
print("LANGGRAPH OUTPUT - WEATHER EXAMPLE")
print("=================================================")
print(langgraph_output_1)
print("Final answer:", langgraph_output_1["result"])
print("\n")

LANGGRAPH OUTPUT - WEATHER EXAMPLE
{'user_input': 'What is the weather in Seattle?', 'category': 'weather', 'result': 'The weather in Seattle is cloudy, 61F.'}
Final answer: The weather in Seattle is cloudy, 61F.




In [15]:
langgraph_output_2 = langgraph_app.invoke({
    "user_input": "Tell me what LangGraph is.",
    "category": "",
    "result": ""
})

print("=================================================")
print("LANGGRAPH OUTPUT - GENERAL EXAMPLE")
print("=================================================")
print(langgraph_output_2)
print("Final answer:", langgraph_output_2["result"])
print("\n")

LANGGRAPH OUTPUT - GENERAL EXAMPLE
{'user_input': 'Tell me what LangGraph is.', 'category': 'general', 'result': 'General response: Tell me what LangGraph is.'}
Final answer: General response: Tell me what LangGraph is.




## Summary

In [16]:
print("=================================================")
print("SUMMARY")
print("=================================================")
print("LangChain:")
print("- Higher-level framework")
print("- Quick way to build agents")
print("- Good for tools, prompts, model integrations")
print()
print("LangGraph:")
print("- Lower-level orchestration framework")
print("- Uses state, nodes, and edges")
print("- Better for complex, multi-step, stateful workflows")
print()
print("Rule of thumb:")
print("- Start with LangChain for simple agents")
print("- Use LangGraph when you need control, branching, loops, or workflow state")

SUMMARY
LangChain:
- Higher-level framework
- Quick way to build agents
- Good for tools, prompts, model integrations

LangGraph:
- Lower-level orchestration framework
- Uses state, nodes, and edges
- Better for complex, multi-step, stateful workflows

Rule of thumb:
- Start with LangChain for simple agents
- Use LangGraph when you need control, branching, loops, or workflow state
